# Standardizing Ground Sampling Distance (GSD)

Orthomosaics from different surveys may have different spatial resolutions (GSD), meaning each pixel represents a different ground area. This directly affects tile coverage, tile count, and model consistency.

To ensure that each tile represents a comparable real-world area and to reduce unnecessary computational load, all orthomosaics are resampled to a common target GSD (e.g., 0.75 cm/pixel) before tiling. This provides consistent spatial context for microsite classification and improves training stability across datasets.

# Resample with Rasterio (preserve georeferencing)

In [1]:
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine

def resample_ortho_to_target_gsd(
    input_path,
    output_path,
    target_gsd_m=0.0060  # 0.60 cm = 0.0060 m
):
    with rasterio.open(input_path) as src:
        current_gsd = src.res[0]

        print(f"Current GSD: {current_gsd*100:.3f} cm/pixel")
        print(f"Target GSD : {target_gsd_m*100:.3f} cm/pixel")

        scale_factor = target_gsd_m / current_gsd
        print(f"Scale factor: {scale_factor:.2f}")

        new_width = int(src.width / scale_factor)
        new_height = int(src.height / scale_factor)

        print(f"New size: {new_width} x {new_height}")

        data = src.read(
            out_shape=(
                src.count,
                new_height,
                new_width
            ),
            resampling=Resampling.bilinear  # good for RGB imagery
        )

        # Update transform
        new_transform = src.transform * Affine.scale(scale_factor)

        profile = src.profile
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform
        })

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(data)

    print("Resampled ortho saved to:", output_path)

In [2]:
resample_ortho_to_target_gsd(
    input_path="./data_set/orthomosaic/A10Segment1.tif",
    output_path="./data_set/orthomosaic/A10Segment1_resampled.tif",
    target_gsd_m=0.0060
)

Current GSD: 0.154 cm/pixel
Target GSD : 0.600 cm/pixel
Scale factor: 3.90
New size: 9912 x 19628
Resampled ortho saved to: ./data_set/orthomosaic/A10Segment1_resampled.tif
